In [ ]:
pip install boto3

In [ ]:
import boto3
import gzip
import pandas as pd
from datetime import datetime, timedelta
import os
from botocore import UNSIGNED
from botocore.config import Config

In [ ]:
# Initialize a session using your credentials
session = boto3.Session(
  aws_access_key_id='f3b9fcba-1401-417e-8aab-20273662677a',
  aws_secret_access_key='5YHF89FZJvHsGBkh_4YVyv7rWiIcoKpr',
)

In [ ]:
# s3 = session.client(
#   's3',
#   endpoint_url='https://files.massive.com',
#   config=Config(signature_version='s3v4'),
# )

# Anonymous S3 client
s3 = boto3.client(
    "s3",
    endpoint_url="https://files.massive.com",
    config=Config(signature_version=UNSIGNED)
)

In [ ]:
# paginator = s3.get_paginator('list_objects_v2')

In [ ]:
# Choose the appropriate prefix depending on the data you need:
# - 'global_crypto' for global cryptocurrency data
# - 'global_forex' for global forex data
# - 'us_indices' for US indices data
# - 'us_options_opra' for US options (OPRA) data
# - 'us_stocks_sip' for US stocks (SIP) data
# prefix = 'us_stocks_sip'  # Example: Change this prefix to match your data need

In [ ]:
# List objects using the selected prefix
# for page in paginator.paginate(Bucket='flatfiles', Prefix=prefix):
#   for obj in page['Contents']:
#     print(obj['Key'])

In [ ]:
bucket_name = "flatfiles"
# bucket_name = "bigdata-stocks"
tickers = ["AAPL", "GOOGL"]

month_folder = "data/month"
year_folder = "data/year"

temp_folder = "temp_downloads"

os.makedirs(month_folder, exist_ok=True)
os.makedirs(year_folder, exist_ok=True)
os.makedirs(temp_folder, exist_ok=True)

In [ ]:
today = datetime.today()

month_start = today - timedelta(days=30)
year_start = today - timedelta(days=365)

In [ ]:
def process_file(local_path, save_folder):
    
    with gzip.open(local_path, "rt") as f:
        df = pd.read_csv(f)

    df = df[df["symbol"].isin(tickers)]

    for ticker in tickers:

        ticker_df = df[df["symbol"] == ticker]

        if len(ticker_df) == 0:
            continue

        ticker_dir = f"{save_folder}/{ticker}"
        os.makedirs(ticker_dir, exist_ok=True)

        file_name = os.path.basename(local_path).replace(".csv.gz", f"_{ticker}.csv")

        ticker_df.to_csv(f"{ticker_dir}/{file_name}", index=False)

In [ ]:
current = year_start

while current <= today:

    date_str = current.strftime("%Y-%m-%d")
    year = current.strftime("%Y")
    month = current.strftime("%m")

    object_key = f"us_stocks_sip/trades_v1/{year}/{month}/{date_str}.csv.gz"

    local_file = f"{temp_folder}/{date_str}.csv.gz"

    try:

        print(f"Downloading {object_key}")

        s3.download_file(bucket_name, object_key, local_file)

        # Decide destination
        if current >= month_start:
            process_file(local_file, month_folder)

        process_file(local_file, year_folder)

        os.remove(local_file)

    except Exception as e:

        print(f"Skipping {date_str} - {e}")

    current += timedelta(days=1)

In [ ]:
# Copy example
# Specify the bucket name

# bucket_name = 'flatfiles'

# # Specify the S3 object key name
# object_key = 'us_stocks_sip/trades_v1/2025/11/2025-11-05.csv.gz'

# # Specify the local file name and path to save the downloaded file
# # This splits the object_key string by '/' and takes the last segment as the file name
# local_file_name = object_key.split('/')[-1]

# # This constructs the full local file path
# local_file_path = './' + local_file_name

In [ ]:
# Download the file
# s3.download_file(bucket_name, object_key, local_file_path)